# 04 · Aspects: what customers talk about, and how they feel about each

**What this does.** It goes beyond "this review is negative" to **"the food was praised, the wait was criticised, and here are the exact words"**. It runs in three steps:
1. **Find what reviews talk about**, the *aspects* (food, staff, price, filters, ice makers…). They are discovered from the reviews themselves, not from a hand-written list, so the same code works for a salon or a bakery.
2. **Score each mention:** a model trained for aspect sentiment reads the part of the sentence about that aspect and says positive, neutral or negative.
3. **Keep the cause:** the exact clause the customer wrote ("we waited 50 minutes"), cut from the review by character position, so it's never a paraphrase.

**Checked against people.** 200 aspect mentions were labelled by hand and compared with the model's answers.

Everything here runs on the laptop CPU, on samples: about 3,000 reviews per domain to build the aspect lists and 300 per domain to score.

In [1]:
import html
import time

import pandas as pd
import spacy
from IPython.display import HTML, display

from review_intel import config
from review_intel.aspects.absa import aspect_mentions, local_absa, local_embedder
from review_intel.aspects.catalogue import build_catalogue, load_catalogue, match, save_catalogue
from review_intel.aspects.phrases import extract_phrases
from review_intel.models.metrics import evaluate, save_metrics

pd.set_option("display.max_colwidth", 140)
df = pd.read_parquet(config.INTERIM_DIR / "reviews.parquet")
nlp = spacy.load("en_core_web_sm", disable=["ner"])
embed = local_embedder()
CATALOGUE_SAMPLE, SCORE_SAMPLE = 3000, 300

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 1. Finding the things reviews talk about

spaCy (a language-analysis library) finds the **nouns** in each sentence and keeps two-word ones together (*customer service*, *wait time*, *ice maker*). It drops:
- pronouns;
- vague words (*thing*, *experience*);
- people, times and the shop itself (*my wife*, *Friday night*, *the restaurant*), which are context rather than something a shop is rated on.

Here is one review with the phrases it found highlighted:

In [2]:
example = df[(df["domain"] == "restaurant") & (df["text"].str.len().between(200, 420))].iloc[3]
found = extract_phrases(example["review_id"], example["text"], nlp)
marked, last = [], 0
for p in found:
    marked += [html.escape(example["text"][last:p["start"]]), f"<mark>{html.escape(example['text'][p['start']:p['end']])}</mark>"]
    last = p["end"]
marked.append(html.escape(example["text"][last:]))
display(HTML("<p style='max-width:760px;line-height:1.6'>" + "".join(marked) + "</p>"))
print([p["text"] for p in found])

[Output removed before publishing: it shows review text, which the Yelp and Amazon licences don't allow us to republish. Run the notebook locally to see it.]


['food', 'food', 'portion', 'worker', 'food', 'room', 'paleta', 'food', 'cherokee street', 'vibe']


## 2. Grouping phrases into aspects (the catalogue)

The same idea is written many ways: *waiter, waitress, hostess*; *price, cost, pricing*. A small sentence-embedding model (**bge-small**) turns each phrase into numbers that place similar meanings close together. Phrases closer than a set distance are grouped, and each group is named after its most common phrase.

These **catalogues** are saved (one per domain, plus a general one for other shop types such as salons) and reused. Every shop's reviews are matched against them, so aspect names stay the same across shops, which the app's Compare page and alerts depend on. The thresholds were tuned on real reviews:
- **Grouping distance 0.25.** At 0.3, *food* already merged with *restaurant* and *dinner*.
- **Matching similarity 0.80.** Unrelated words still score about 0.70 with this model.

In [3]:
start = time.time()
phrases_by_domain = {}
for domain in config.DOMAINS:
    part = df[(df["domain"] == domain) & (df["split"] == "train")].sample(CATALOGUE_SAMPLE, random_state=config.SEED)
    phrases_by_domain[domain] = [p for rid, text in zip(part["review_id"], part["text"], strict=True) for p in extract_phrases(rid, text, nlp)]
    print(f"{domain}: {len(phrases_by_domain[domain]):,} phrases from {len(part):,} reviews")
catalogues = {d: build_catalogue(ph, embed, min_count=25, max_aspects=15, domain=d) for d, ph in phrases_by_domain.items()}
all_phrases = [p for ph in phrases_by_domain.values() for p in ph]
catalogues["general"] = build_catalogue(all_phrases, embed, min_count=60, max_aspects=20, domain="general")
for name, cat in catalogues.items():
    save_catalogue(name, cat)
print(f"Built and saved {len(catalogues)} catalogues in {time.time() - start:,.0f} s")

restaurant: 46,621 phrases from 3,000 reviews


grocery: 16,018 phrases from 3,000 reviews


appliances: 17,097 phrases from 3,000 reviews


Built and saved 4 catalogues in 59 s


In [4]:
for name, cat in catalogues.items():
    print(f"\n{name.upper()}")
    display(pd.DataFrame([{"aspect": a["name"], "mentions": a["count"], "includes": ", ".join(a["examples"])} for a in cat]))


RESTAURANT


,aspect,mentions,includes
0,Food,2524,"food, meal, dinner, entree, cuisine, dine"
1,Service,915,"service, serving, serve, table service"
2,Drink,457,"drink, soda, alcohol, beverage, drinking, drinker"
3,Menu,432,menu
4,Pizza,412,"pizza, pizza place, mozzarella, pepperoni, pepperoni pizza, pizza hut"
5,Staff,409,"staff, staff member, kitchen staff, service staff"
6,Table,405,"table, opentable, table side, picnic table, community table"
7,Bar,401,"bar, bartender, bar area, bar food, sport bar, wine bar"
8,Dish,377,"dish, shrimp, shrimp dish, chicken dish, fish dish"
9,Meat,376,"meat, steak, beef, ground beef, beef shank"



GROCERY


,aspect,mentions,includes
0,Flavor,689,"flavor, flavoring, flavour, flavor packet, smokey flavor, type flavor"
1,Taste,518,"taste, tasting, taste bud, taste test, chemical taste, taste difference"
2,Coffee,363,"coffee, coffee drinker, coffee maker, coffee lover, coffee drink, coffee flavor"
3,Tea,265,"tea, thai tea, tea lover, tea taste, yogi tea, taste tea"
4,Price,264,"price, cost, pricing, price point, priced, price change"
5,Bag,227,"bag, tea bag, purse, size bag, teabag, goody bag"
6,Box,174,"box, shipping box, delivery box, amazon box, size box, board box"
7,Package,138,"package, pack, sample, packages, sample pack, sampler pack"
8,Brand,130,"brand, store brand, seeds brand, brand name"
9,Chocolate,125,"chocolate, chocolate flavor, milk chocolate, chocolate taste, chocolate milk, chocolatey"



APPLIANCES


,aspect,mentions,includes
0,Filter,685,"filter, water filter, paper filter, coffee filter, brand filter, filter bag"
1,Price,407,"price, cost, price point, price gouging, price way, price tag"
2,Water,342,"water, drinking water, water supply, water quality, water system, water source"
3,Fridge,290,"fridge, refrigerator, freezer, refrigerator water, fridge water, profile refrigerator"
4,Coffee,267,"coffee, coffee taste, starbucks, coffe, coffee flavor, coffee make"
5,Problem,255,"problem, issue, design issue, annoying problem"
6,Ice,251,"ice, ice cube, ice bin, ice basket, ice ice baby, size ice cube"
7,Replacement,247,"replacement, replacement part, replacement filter, filter replacement, replacement ice maker, replacement item"
8,Work,243,"work, working, work grest, work soooo, worke"
9,Dryer,237,"dryer, dryer vent, dryer connector, dryer clip, dryer lol, dryer set"



GENERAL


,aspect,mentions,includes
0,Food,2678,"food, meal, dinner, diner, dining, dining room"
1,Service,1041,"service, phone, call, contact, service call, phone call"
2,Price,994,"price, cost, pricing, price point, price tag"
3,Flavor,982,"flavor, flavorless, flavoring, flavour, flavor profile, flavor packet"
4,Coffee,836,"coffee, coffee maker, coffee ground, ground coffee, cup coffee maker, morning coffee"
5,Taste,766,"taste, tasting, taste bud, water taste, tastebud, chlorine taste"
6,Filter,634,"filter, water filter, paper filter, coffee filter, brand filter, air filter"
7,Water,496,"water, boiling water, hydration, drinking water, watering"
8,Drink,471,"drink, alcohol, liquor, drinking, rum, vodka"
9,Work,467,"work, job, working"


## 3. Scoring each mention

For 300 unseen test reviews per domain:
1. Each phrase is matched to its domain's catalogue.
2. The **clause** around each match becomes the cause.
3. The aspect-sentiment model (`yangheng/deberta-v3-base-absa-v1.1`, trained on restaurant and laptop reviews) scores it.

The model reads the clause rather than the whole sentence, so *"the biryani was great **but** we waited 50 minutes"* gives food a positive and the wait a negative.

In [5]:
absa = local_absa()
start = time.time()
rows = []
for domain in config.DOMAINS:
    cat = load_catalogue(domain)
    names = {a["id"]: a["name"] for a in cat}
    part = df[(df["domain"] == domain) & (df["split"] == "test")].sample(SCORE_SAMPLE, random_state=config.SEED)
    for rid, text in zip(part["review_id"], part["text"], strict=True):
        matched = match(extract_phrases(rid, text, nlp), cat, embed)
        for m in aspect_mentions(rid, text, matched, names, absa):
            rows.append({**m, "domain": domain, "aspect": names[m["aspectId"]], "cause": text[m["start"]:m["end"]]})
mentions = pd.DataFrame(rows)
print(f"{len(mentions):,} aspect mentions in {3 * SCORE_SAMPLE} reviews, scored in {time.time() - start:,.0f} s")
summary = (mentions.groupby(["domain", "aspect"])["sentiment"].value_counts(normalize=True).unstack(fill_value=0) * 100)
summary["mentions"] = mentions.groupby(["domain", "aspect"]).size()
summary = summary[["mentions", "positive", "neutral", "negative"]].round(0).astype(int)
summary.sort_values(["domain", "mentions"], ascending=[True, False])

C:\dev\review-intel\pipeline\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

2,087 aspect mentions in 900 reviews, scored in 116 s


sentiment               mentions  positive  neutral  negative
domain     aspect                                            
appliances Filter             58        31       41        28
           Fridge             49        14       65        20
           Price              48        83       15         2
           Washer             48        35       52        12
           Problem            36        33        6        61
           Water              34        53       32        15
           Coffee             30        33       57        10
           Dryer              30        27       43        30
           Ice                30        30       53        17
           Quality            25        80        4        16
           Fit                23        87        0        13
           Machine            22        23       32        45
           Work               22        86        9         5
           Replacement        21        62       33         5
           Cup                16        50       44         6
grocery    Flavor             86        67       10        22
           Coffee             72        51       36        12
           Taste              49        57        6        37
           Delicious          39        97        3         0
           Tea                33        64       24        12
           Price              31        71        3        26
           Bag                30        10       47        43
           Package            26        35       50        15
           Sugar              19        16       37        47
           Cup                18        39       39        22
           Chocolate          17        29       47        24
           Box                15        20       67        13
           Brand              15        60       33         7
           Candy              14        57       21        21
           Quality            10        80       10        10
restaurant Food              373        46       43        12
           Service            93        53       10        38
           Drink              89        40       47        12
           Cheese             71        37       52        11
           Meat               69        42       35        23
           Sauce              60        43       35        22
           Price              51        57       18        25
           Menu               44        25       66         9
           Bar                42        40       50        10
           Dish               42        55       33        12
           Pizza              41        51       39        10
           Staff              40        68       12        20
           Waitress           38        39       18        42
           Table              35        14       77         9
           Server             33        36       15        48

## 4. Causes: the customer's own words

Here is one example of each sentiment per domain. The cause is cut straight out of the review, with nothing reworded.

In [6]:
ex = mentions.groupby(["domain", "sentiment"]).sample(1, random_state=config.SEED)
ex[["domain", "aspect", "sentiment", "confidence", "cause"]].round(2)

[Output removed before publishing: it shows review text, which the Yelp and Amazon licences don't allow us to republish. Run the notebook locally to see it.]


## 5. Checking it against hand-written labels

200 mentions were drawn, spread across domains and across the model's three answers. Each was labelled **by reading the cause**: is the customer positive, neutral or negative about *this aspect* here? The labels were written by Claude, reading each clause; the labels are in `pipeline/reports/validation/absa_pairs.csv`. The clauses they judge are review text, so they stay on the laptop (`data/processed/absa_pairs_with_text.csv`, gitignored) and aren't published. Then the model's answers are scored against them, just like the sentiment models.

In [7]:
# Labels only (committed); the clauses they judge are review text, so they stay local (licence).
VALID = config.REPORTS_DIR / "validation" / "absa_pairs.csv"
TEXT = config.PROCESSED_DIR / "absa_pairs_with_text.csv"
if not VALID.exists() and not TEXT.exists():
    per = 200 // (len(config.DOMAINS) * 3) + 1
    # (a loop, not groupby().apply: pandas 3 leaves the grouping columns out of apply's groups)
    pick = pd.concat([g.sample(min(len(g), per), random_state=config.SEED) for _, g in mentions.groupby(["domain", "sentiment"])])
    pick = pick.head(200).reset_index(drop=True)
    TEXT.parent.mkdir(parents=True, exist_ok=True)
    pick.assign(pair_id=range(len(pick)), gold="", labelled_by="")[["pair_id", "domain", "aspect", "cause", "sentiment", "gold", "labelled_by"]].to_csv(TEXT, index=False)
    print(f"Wrote {len(pick)} pairs to label → {TEXT.relative_to(config.REPO_ROOT)}")
if VALID.exists():
    pairs = pd.read_csv(VALID, keep_default_na=False)
    if TEXT.exists():
        pairs = pairs.merge(pd.read_csv(TEXT, keep_default_na=False)[["pair_id", "cause"]], on="pair_id", how="left")
else:
    pairs = pd.read_csv(TEXT, keep_default_na=False)
labelled = pairs[pairs["gold"] != ""]
if len(labelled) < len(pairs):
    print(f"{len(labelled)} of {len(pairs)} pairs labelled so far; validation runs once all are labelled.")
else:
    if not VALID.exists():
        VALID.parent.mkdir(parents=True, exist_ok=True)
        pairs.drop(columns=["cause"]).to_csv(VALID, index=False)
    v = evaluate(labelled["gold"], labelled["sentiment"])
    save_metrics("absa_validation", {**v, "labelled_by": sorted(set(labelled["labelled_by"]))}, test_size=len(labelled))
    print(f"Aspect sentiment vs. hand labels: macro F1 {v['macro_f1']:.3f} · accuracy {v['accuracy']:.3f} on {len(labelled)} pairs")
    display(pd.DataFrame(v["per_class"]).T.round(3))
    display(pd.DataFrame(v["confusion"], index=[f"true {l}" for l in config.LABELS], columns=[f"model {l}" for l in config.LABELS]))
    disagree = labelled[labelled["gold"] != labelled["sentiment"]]
    display(disagree[[c for c in ["domain", "aspect", "cause", "sentiment", "gold"] if c in disagree]].head(12))

Aspect sentiment vs. hand labels: macro F1 0.724 · accuracy 0.725 on 200 pairs


,precision,recall,f1,support
negative,0.768,0.757,0.763,70.0
neutral,0.565,0.639,0.600,61.0
positive,0.855,0.768,0.809,69.0


,model negative,model neutral,model positive
true negative,53,16,1
true neutral,14,39,8
true positive,2,14,53


[Output removed before publishing: it shows review text, which the Yelp and Amazon licences don't allow us to republish. Run the notebook locally to see it.]


## Key findings

1. **The catalogues group the same idea well.** *waiter, waitress, hostess* became one aspect; *price, cost, pricing, price point* another; *filter, water filter, replacement filter* a third. Each shop type got 15 aspects, and the general catalogue (for shop types we have no data for) 20.
2. **Some aspects are too specific or aren't really aspects.** Restaurants have **Pizza** and **Cheese** as aspects next to **Food**, and **Waitress** and **Server** separately from **Staff**, because the grouping threshold (0.25) is kept tight so that *food* and *restaurant* stay apart. Grocery has **Delicious** (an opinion word spaCy tagged as a noun), and appliances have **Work** and **Problem**. These are useful signals but not things a shop can fix. Worth a stop-list in a later pass.
3. **Mentions are plentiful.** 2,087 aspect mentions in 900 unseen reviews (about 2.3 per review), scored on the laptop CPU in 116 s.
4. **The aspect verdicts look right in aggregate.** Appliances **Problem** is 61% negative while **Fit** and **Work** are over 85% positive; grocery **Delicious** is 97% positive and **Bag** 43% negative (torn and stale bags). That's the kind of contrast the owner pages need.
5. **Against 200 hand-written labels: macro F1 0.724, accuracy 72.5%.** The labels were written by Claude reading each cause clause, not by an independent person, so treat this as a sanity check rather than a final score. The labels are in `reports/validation/absa_pairs.csv`; the clauses stay local because they are review text.
6. **Errors are almost all about neutral, not flipped verdicts.** Only 1 negative was called positive and 2 positives negative. Neutral is the weakest class (F1 0.60). Typical mistakes: a review saying a replacement part ended the problems (the word "problem" pulls it towards negative), and plain instructions, such as how much water to add, that only some readers would call neutral.
7. **Next:** notebook 05 puts all of this together into one shop's report, now with DistilBERT for the per-review sentiment.